In [1]:
from google import genai
from google.genai import types
import io
import httpx
import os

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import time
from google import genai
from google.genai import types
import io
import httpx

client = genai.Client()
long_context_pdf_path = "https://www.nasa.gov/wp-content/uploads/static/history/alsj/a17/A17_FlightPlan.pdf"

print("--- Step 1: Uploading File ---")
# Retrieve and upload the PDF using the File API
doc_io = io.BytesIO(httpx.get(long_context_pdf_path).content)
document = client.files.upload(
  file=doc_io,
  config=dict(mime_type='application/pdf'))

model_name = "gemini-3-flash-preview"
system_instruction = "You are an expert analyzing transcripts."

print("\n--- Step 2: Creating Cache ---")
# Start Timer for Cache Creation
start_cache = time.time()

cache = client.caches.create(
    model=model_name,
    config=types.CreateCachedContentConfig(
      system_instruction=system_instruction,
      contents=[document],
    ))

end_cache = time.time()
print(f"Cache created in: {end_cache - start_cache:.2f} seconds")
print(f'{cache=}')

print("\n--- Step 3: First Generation (No Cache) ---")
# Start Timer for First Request
start_gen_1 = time.time()

response = client.models.generate_content(
  model=model_name,
  contents=[document, "Please summarize this transcript"]
)

end_gen_1 = time.time()
print(f"1st Response generated in: {end_gen_1 - start_gen_1:.2f} seconds")

print("\n--- Step 4: 2nd Generation (Using Cache) ---")
# Start Timer for First Request
start_gen_2 = time.time()

response = client.models.generate_content(
  model=model_name,
  contents="Please summarize this transcript",
  config=types.GenerateContentConfig(
    cached_content=cache.name
  ))

end_gen_2 = time.time()
print(f"2nd Response generated in: {end_gen_2 - start_gen_2:.2f} seconds")
# print(response.text) # Hiding text to keep output clean

print("\n--- Step 5: 3rd Generation (Using Cache) ---")
# Start Timer for Second Request
# This simulates a follow-up question in a chat interface or a loop.
start_gen_3 = time.time()

response_3 = client.models.generate_content(
  model=model_name,
  contents="Please summarize this transcript",
  config=types.GenerateContentConfig(
    cached_content=cache.name
  ))

end_gen_3 = time.time()
print(f"3rd Response generated in: {end_gen_3 - start_gen_3:.2f} seconds")

# Optional: Cleanup to stop paying for storage
# client.caches.delete(name=cache.name)

--- Step 1: Uploading File ---

--- Step 2: Creating Cache ---
Cache created in: 2.80 seconds
cache=CachedContent(
  create_time=datetime.datetime(2025, 12, 19, 22, 17, 5, 889082, tzinfo=TzInfo(0)),
  display_name='',
  expire_time=datetime.datetime(2025, 12, 19, 23, 17, 4, 663666, tzinfo=TzInfo(0)),
  model='models/gemini-3-flash-preview',
  name='cachedContents/0q7g8gq8b5rrt0quwarbojwn5qzh83qb0g6tm4oa',
  update_time=datetime.datetime(2025, 12, 19, 22, 17, 5, 889082, tzinfo=TzInfo(0)),
  usage_metadata=CachedContentUsageMetadata(
    total_token_count=346089
  )
)

--- Step 3: First Generation (No Cache) ---
1st Response generated in: 42.25 seconds

--- Step 4: 2nd Generation (Using Cache) ---
2nd Response generated in: 31.77 seconds

--- Step 5: 3rd Generation (Using Cache) ---
3rd Response generated in: 40.85 seconds


In [4]:
for cache in client.caches.list():
  print(cache)
  print(cache.name)
  client.caches.delete(name=cache.name)

name='cachedContents/0q7g8gq8b5rrt0quwarbojwn5qzh83qb0g6tm4oa' display_name='' model='models/gemini-3-flash-preview' create_time=datetime.datetime(2025, 12, 19, 22, 17, 5, 889082, tzinfo=TzInfo(0)) update_time=datetime.datetime(2025, 12, 19, 22, 17, 5, 889082, tzinfo=TzInfo(0)) expire_time=datetime.datetime(2025, 12, 19, 23, 17, 4, 663666, tzinfo=TzInfo(0)) usage_metadata=CachedContentUsageMetadata(
  total_token_count=346089
)
cachedContents/0q7g8gq8b5rrt0quwarbojwn5qzh83qb0g6tm4oa
